# Séance 8 · Projet Kaggle Titanic (2/2) : améliorer le score et présenter · ⭐⭐⭐

**Niveau : ⭐⭐⭐ Avancé**

La semaine dernière, vous avez fait votre première soumission. Aujourd'hui, on passe en mode compétition : comparer plusieurs modèles, comprendre **pourquoi** l'un marche mieux que l'autre, éviter le piège du sur-apprentissage, régler les boutons du modèle, et faire la soumission finale.

Ce notebook tourne dans **Google Colab** (rien à installer), `Maj + Entrée` pour exécuter chaque cellule. Toujours en **binôme**, on échange les rôles toutes les 20 minutes.

**Livrable de la séance** : une soumission finale sur Kaggle, le tableau des scores du groupe, et une présentation de 5 minutes par binôme.


## Préparation

On recharge les données et on **reprend la fonction `preparer()`** de la séance 7 (si vous y aviez ajouté une variable à la Question 1, recopiez-la ici). N'oublie pas de redéposer `train.csv` et `test.csv` dans le panneau Fichiers de Colab.

In [ ]:
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import cross_val_score, cross_validate, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

sns.set_theme(style="whitegrid")
print("Prêt !")

In [ ]:
URL_SECOURS = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"

if os.path.exists("train.csv"):
    train = pd.read_csv("train.csv")
    print("train.csv de Kaggle chargé")
else:
    try:
        train = pd.read_csv(URL_SECOURS)
        print("Copie publique chargée (mêmes colonnes que train.csv). Pour la vraie compétition, dépose train.csv dans Colab.")
    except Exception as erreur:
        print("Impossible de charger les données : pas de réseau ?", erreur)
        raise

# test.csv n'a pas la colonne Survived : c'est à nous de la prédire
test = pd.read_csv("test.csv") if os.path.exists("test.csv") else None
print("test.csv :", "chargé" if test is not None else "absent (tout fonctionne quand même, sauf la soumission)")
train.head()

In [ ]:
def regrouper_titre(titre):
    """Garde les 4 titres fréquents, regroupe le reste dans « Autre »."""
    if titre in ["Mr", "Mrs", "Miss", "Master"]:
        return titre
    if titre in ["Mlle", "Ms"]:
        return "Miss"
    if titre == "Mme":
        return "Mrs"
    return "Autre"          # Dr, Rev, Col, Major, Countess, Capt...


COLONNES = ["Pclass", "Sex", "Age", "Fare", "Embarked", "Famille", "Seul", "Titre"]

def preparer(df):
    """Transforme le tableau brut de Kaggle en tableau de nombres prêt pour un modèle."""
    d = df.copy()
    # 1. Nouvelles variables
    d["Famille"] = d["SibSp"] + d["Parch"] + 1
    d["Seul"] = (d["Famille"] == 1).astype(int)
    d["Titre"] = d["Name"].str.extract(r",\s*([^\.]+)\.")[0].str.strip().apply(regrouper_titre)
    # 2. Cases vides : l'âge médian de chaque titre (un « Master » a 4 ans, un « Mr » 30)
    d["Age"] = d.groupby("Titre")["Age"].transform(lambda s: s.fillna(s.median()))
    d["Age"] = d["Age"].fillna(d["Age"].median())
    d["Fare"] = d["Fare"].fillna(d["Fare"].median())
    d["Embarked"] = d["Embarked"].fillna("S")
    # 3. Tout en nombres
    d["Sex"] = (d["Sex"] == "female").astype(int)
    d["Embarked"] = d["Embarked"].map({"S": 0, "C": 1, "Q": 2})
    d["Titre"] = d["Titre"].map({"Mr": 0, "Mrs": 1, "Miss": 2, "Master": 3, "Autre": 4})
    return d[COLONNES]


X = preparer(train)
y = train["Survived"]
print(X.shape, "-", X.isna().sum().sum(), "case vide")
X.head(3)

## 1. Comparer 4 modèles dans un tableau

Quatre familles de modèles, quatre façons de raisonner :
- **Régression logistique** : une balance qui pèse chaque indice.
- **Arbre de décision** : des questions en cascade (« femme ? » → « 3e classe ? » → ...). Lisible, mais fragile.
- **Forêt aléatoire** : 200 arbres différents qui votent. Plus stable.
- **Gradient boosting** : des petits arbres construits **l'un après l'autre**, chacun corrigeant les erreurs du précédent. Comme réviser en refaisant surtout les exercices qu'on a ratés.

On les mesure tous avec la même validation croisée (5 paquets) et on range les résultats dans un tableau.

In [ ]:
modeles = {
    "Régression logistique": LogisticRegression(max_iter=1000),
    "Arbre de décision": DecisionTreeClassifier(max_depth=4, random_state=42),
    "Forêt aléatoire": RandomForestClassifier(n_estimators=200, max_depth=5, random_state=42),
    "Gradient boosting": GradientBoostingClassifier(n_estimators=100, max_depth=3, random_state=42),
}

resultats = []
for nom, modele in modeles.items():
    scores = cross_val_score(modele, X, y, cv=5)
    resultats.append({"Modèle": nom, "Exactitude moyenne": scores.mean(), "Écart-type": scores.std()})

tableau = pd.DataFrame(resultats).sort_values("Exactitude moyenne", ascending=False).round(3)
tableau

In [ ]:
plt.figure(figsize=(7, 3.5))
plt.barh(tableau["Modèle"], tableau["Exactitude moyenne"], xerr=tableau["Écart-type"], color="tab:blue")
plt.xlim(0.7, 0.9)
plt.xlabel("Exactitude (validation croisée)")
plt.title("Les 4 modèles se tiennent dans un mouchoir de poche")
plt.gca().invert_yaxis()
plt.show()

L'**écart-type** (la petite barre noire) dit si le modèle est **stable** : un score qui varie de 76 % à 86 % selon le paquet est moins fiable qu'un 81 % constant. Sur Kaggle, un écart de 1 % entre deux modèles peut n'être que du hasard.

**Exercice** : ajoute un 5e modèle, les k plus proches voisins (`from sklearn.neighbors import KNeighborsClassifier`, `n_neighbors=5`). Pourquoi est-il si mauvais ici ? (indice : `Fare` vaut jusqu'à 512, `Sex` vaut 0 ou 1)

<details><summary>Solution</summary>

```python
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

voisins = KNeighborsClassifier(n_neighbors=5)
print("k-NN brut :", round(cross_val_score(voisins, X, y, cv=5).mean() * 100, 1), "%")
# Il mesure des distances : une différence de 100 livres écrase une différence homme/femme.
# Solution : mettre toutes les colonnes à la même échelle avant.
voisins_ok = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5))
print("k-NN mis à l'échelle :", round(cross_val_score(voisins_ok, X, y, cv=5).mean() * 100, 1), "%")
```
</details>

In [ ]:
# À toi

## 2. Pourquoi l'un marche mieux ? L'importance des variables

Une forêt (ou un gradient boosting) sait dire quelles variables elle a le plus utilisées pour trancher : c'est l'**importance des variables**. Un score sans explication ne vaut rien devant un « client » : ce graphique est votre meilleur argument.

In [ ]:
foret = RandomForestClassifier(n_estimators=200, max_depth=5, random_state=42).fit(X, y)
boosting = GradientBoostingClassifier(n_estimators=100, max_depth=3, random_state=42).fit(X, y)

importances = pd.DataFrame({
    "Forêt aléatoire": foret.feature_importances_,
    "Gradient boosting": boosting.feature_importances_,
}, index=COLONNES).sort_values("Forêt aléatoire")

importances.plot(kind="barh", figsize=(7, 4))
plt.title("Quelles variables font la décision ?")
plt.xlabel("Importance (somme = 1)")
plt.show()

`Titre` et `Sex` racontent la même histoire (« les femmes d'abord ») et se partagent l'importance. `Fare` et `Pclass` aussi. La forêt étale ses choix, le boosting se concentre sur 2-3 variables : c'est pour ça que leurs scores diffèrent un peu.

**Exercice** : retire la variable la plus importante (par exemple `X.drop(columns=["Titre"])`) et remesure la forêt. Le score s'effondre-t-il ? Pourquoi pas ?

<details><summary>Solution</summary>

```python
X_sans_titre = X.drop(columns=["Titre"])
print(round(cross_val_score(foret, X_sans_titre, y, cv=5).mean() * 100, 1), "%")
# Il baisse à peine : Sex et Age portent presque la même information que Titre.
```
</details>

In [ ]:
# À toi

## 3. Le sur-apprentissage : apprendre par cœur

Imagine un élève qui apprend par cœur les annales : 100 % sur les vieux sujets, mais perdu devant un sujet nouveau. Un modèle trop compliqué fait pareil : il **mémorise** les 891 passagers au lieu de comprendre la règle. C'est le **sur-apprentissage** (*overfitting*).

Pour le voir, on fait grandir un arbre de décision de la profondeur 1 à 15 et on mesure deux choses : son exactitude sur les données qu'il a vues (**train**) et sur les paquets cachés (**validation**).

In [ ]:
profondeurs = range(1, 16)
score_train, score_valid = [], []

for p in profondeurs:
    arbre = DecisionTreeClassifier(max_depth=p, random_state=42)
    cv = cross_validate(arbre, X, y, cv=5, return_train_score=True)
    score_train.append(cv["train_score"].mean())
    score_valid.append(cv["test_score"].mean())

plt.figure(figsize=(8, 4))
plt.plot(profondeurs, score_train, "o-", label="Train (données vues)")
plt.plot(profondeurs, score_valid, "s-", label="Validation (données cachées)")
plt.xlabel("Profondeur de l'arbre (max_depth)")
plt.ylabel("Exactitude")
plt.title("Plus l'arbre est profond, plus il apprend par cœur")
plt.legend()
plt.show()

meilleure = profondeurs[int(np.argmax(score_valid))]
print("Meilleure profondeur en validation :", meilleure, "→", round(max(score_valid) * 100, 1), "%")

Lecture du graphique :
- À gauche (profondeur 1-2), les deux courbes sont basses : le modèle est **trop simple** (sous-apprentissage).
- Au milieu, la validation atteint son maximum : c'est la **bonne zone**.
- À droite, le train monte vers 98 % pendant que la validation redescend : l'arbre a **appris par cœur**. L'écart entre les deux courbes, c'est le sur-apprentissage.

**Exercice** : trace la même courbe pour une forêt (`RandomForestClassifier(n_estimators=100)`) avec `max_depth` dans `[2, 4, 6, 8, 10, None]`. La forêt sur-apprend-elle autant que l'arbre seul ?

<details><summary>Solution</summary>

```python
prof = [2, 4, 6, 8, 10, None]
tr, va = [], []
for p in prof:
    cv = cross_validate(RandomForestClassifier(n_estimators=100, max_depth=p, random_state=42), X, y, cv=5, return_train_score=True)
    tr.append(cv["train_score"].mean()); va.append(cv["test_score"].mean())
etiquettes = [str(p) for p in prof]
plt.plot(etiquettes, tr, "o-", label="Train"); plt.plot(etiquettes, va, "s-", label="Validation")
plt.legend(); plt.show()
# Le train grimpe aussi, mais la validation tient mieux : 200 arbres qui votent lissent les erreurs de chacun.
```
</details>

In [ ]:
# À toi

## 4. Régler les hyperparamètres avec `GridSearchCV`

Un **hyperparamètre** est un bouton qu'on choisit **avant** l'entraînement (`max_depth`, `n_estimators`...), contrairement aux paramètres que le modèle apprend tout seul. Plutôt que de tourner les boutons à la main, `GridSearchCV` teste **toutes les combinaisons** d'une grille et garde la meilleure, avec validation croisée à chaque fois.

On règle 2 boutons de la forêt : `max_depth` (profondeur) et `min_samples_leaf` (nombre minimum de passagers pour qu'une feuille de l'arbre soit valide, un frein contre le par cœur). 4 × 3 = 12 combinaisons × 5 paquets = 60 entraînements, moins de 20 secondes.

In [ ]:
grille = {"max_depth": [3, 5, 7, 9], "min_samples_leaf": [1, 3, 5]}

recherche = GridSearchCV(RandomForestClassifier(n_estimators=100, random_state=42), grille, cv=5, n_jobs=-1)

debut = time.time()
recherche.fit(X, y)
print(f"12 combinaisons testées en {time.time() - debut:.1f} s")
print("Meilleurs réglages :", recherche.best_params_)
print("Exactitude :", round(recherche.best_score_ * 100, 1), "%")

In [ ]:
# Toutes les combinaisons dans une grille de couleurs
res = pd.DataFrame(recherche.cv_results_)
carte = res.pivot(index="param_max_depth", columns="param_min_samples_leaf", values="mean_test_score")

plt.figure(figsize=(5, 3.5))
sns.heatmap(carte.astype(float), annot=True, fmt=".3f", cmap="YlGn")
plt.title("Exactitude selon les 2 réglages")
plt.show()

**Exercice** : fais la même chose pour le gradient boosting avec `n_estimators` dans `[50, 100, 200]` et `learning_rate` dans `[0.05, 0.1, 0.2]` (la vitesse à laquelle chaque arbre corrige le précédent). Bat-il la forêt ?

<details><summary>Solution</summary>

```python
grille_gb = {"n_estimators": [50, 100, 200], "learning_rate": [0.05, 0.1, 0.2]}
recherche_gb = GridSearchCV(GradientBoostingClassifier(max_depth=3, random_state=42), grille_gb, cv=5, n_jobs=-1)
recherche_gb.fit(X, y)
print(recherche_gb.best_params_, round(recherche_gb.best_score_ * 100, 1), "%")
```
</details>

In [ ]:
# À toi

## 5. Les pièges : fuite de données et score trop beau

Trois pièges classiques qui donnent des scores magnifiques... et faux.

### Piège 1 : la fuite de données (*data leakage*)
Une **fuite**, c'est quand une colonne contient (même déguisée) la réponse. Le modèle triche sans le savoir. Démonstration : on glisse une colonne `Canot` qui vaut... la survie.

In [ ]:
X_fuite = X.copy()
X_fuite["Canot"] = train["Survived"]        # la réponse déguisée en indice

score_fuite = cross_val_score(RandomForestClassifier(n_estimators=100, random_state=42), X_fuite, y, cv=5).mean()
print(f"Avec la fuite : {score_fuite*100:.1f} % → trop beau pour être vrai")
print("Sur test.csv, la colonne Canot n'existe pas : ce modèle est inutilisable.")

### Piège 2 : se noter sur ses propres données
Mesurer un modèle sur les passagers qu'il a vus, c'est corriger son contrôle avec le corrigé sous les yeux.

In [ ]:
foret_profonde = RandomForestClassifier(n_estimators=200, random_state=42).fit(X, y)   # sans limite de profondeur

print("Score sur les données vues     :", round(foret_profonde.score(X, y) * 100, 1), "%")
print("Score en validation croisée    :", round(cross_val_score(foret_profonde, X, y, cv=5).mean() * 100, 1), "%")

### Piège 3 : oublier la règle bête
Avant de se réjouir d'un score, il faut le comparer à la règle la plus simple possible : « toutes les femmes survivent, aucun homme ». C'est exactement le fichier `gender_submission.csv` de Kaggle.

In [ ]:
regle_bete = (train["Sex"] == "female").astype(int)
print("Règle « les femmes survivent » :", round((regle_bete == y).mean() * 100, 1), "% sur train")
print("Sur Kaggle, cette règle donne 0.76555. Ton modèle doit faire mieux, sinon il ne sert à rien.")

Et sur le classement Kaggle ? Un score entre **0.77 et 0.82** est excellent. Au-delà de **0.85**, méfiance : la liste réelle des survivants est publique, certains la copient. Ce n'est pas du machine learning, c'est du copier-coller, et ça n'apprend rien.

**Exercice** : fuite ou pas fuite ? Réponds pour chaque situation, puis compare avec la solution.
1. Ajouter la colonne `Cabine_connue` (la cabine est renseignée ou non).
2. Remplir les âges manquants de `test.csv` avec la médiane calculée sur `train`.
3. Ajouter une colonne « nombre de survivants parmi les gens qui ont le même numéro de billet ».

<details><summary>Solution</summary>

1. **Fuite légère** : les cabines connues sont surtout celles des survivants (ils ont pu témoigner). Ça marche sur `train`, moins bien sur `test`.
2. **Pas de fuite** : on n'utilise que des informations disponibles au moment de prédire. C'est même la bonne pratique.
3. **Fuite franche** : on utilise `Survived` pour fabriquer la variable. Score magnifique sur `train`, impossible à calculer sur `test`.
</details>

In [ ]:
# À toi : écris tes réponses en commentaire
# 1.
# 2.
# 3.

## 6. Projet : la soumission finale (80 min)

En binôme, dans l'ordre :
- **Étape A (25 min)** : choisissez votre modèle final. Utilisez `recherche.best_estimator_`, ou le gradient boosting réglé à l'exercice de la section 4. Essayez d'ajouter ou de retirer une variable dans `preparer()` et remesurez (Question 1).
- **Étape B (15 min)** : générez `submission_finale.csv`, soumettez sur Kaggle, notez le score (Questions 2 et 3).
- **Étape C (15 min)** : remplissez le tableau des scores du groupe (Question 4).
- **Étape D (25 min)** : préparez la présentation de 5 minutes (section 7).

In [ ]:
# Question 1 : votre modèle final. Remplacez par votre meilleur candidat et justifiez en commentaire.
modele_final = recherche.best_estimator_
# Pourquoi ce modèle ? ...

score_final_cv = cross_val_score(modele_final, X, y, cv=5).mean()
print(f"Modèle final : {type(modele_final).__name__}, validation croisée : {score_final_cv*100:.1f} %")

In [ ]:
# Question 2 : la soumission finale
modele_final.fit(X, y)

if test is not None:
    predictions = modele_final.predict(preparer(test))
    submission = pd.DataFrame({"PassengerId": test["PassengerId"], "Survived": predictions})
    submission.to_csv("submission_finale.csv", index=False)
    print("submission_finale.csv créé :", len(submission), "lignes.")
    print("Panneau Fichiers → Télécharger → Kaggle : Submit Prediction")
    print(submission.head())
else:
    print("test.csv absent : dépose-le dans Colab (panneau Fichiers) puis relance cette cellule.")

In [ ]:
# Question 3 : vos deux scores Kaggle (ex. 0.77511)
SCORE_PREMIERE_SOUMISSION = None      # séance 7
SCORE_SOUMISSION_FINALE = None        # aujourd'hui

if None in (SCORE_PREMIERE_SOUMISSION, SCORE_SOUMISSION_FINALE):
    print("Remplace les None par vos scores Kaggle.")
else:
    progres = (SCORE_SOUMISSION_FINALE - SCORE_PREMIERE_SOUMISSION) * 100
    print(f"Première : {SCORE_PREMIERE_SOUMISSION:.5f} → Finale : {SCORE_SOUMISSION_FINALE:.5f} ({progres:+.2f} points)")
    if progres < 0:
        print("Le score a baissé ? Ça arrive : le public leaderboard ne juge que 50 % de test.csv. Gardez la version la plus solide en validation croisée.")

### Question 4 : le tableau du groupe

Chaque binôme annonce ses résultats, tout le monde remplit le tableau (au tableau blanc ou dans la cellule suivante).

| Binôme | Score 1re soumission | Score final | Modèle final | Variable ajoutée |
|---|---|---|---|---|
| Binôme 1 | | | | |
| Binôme 2 | | | | |
| Binôme 3 | | | | |
| Binôme 4 | | | | |

In [ ]:
# Remplacez les exemples par les vrais résultats du groupe
groupe = pd.DataFrame({
    "Binôme": ["Nous", "Binôme 2", "Binôme 3", "Binôme 4"],
    "Première": [0.775, 0.766, 0.770, None],
    "Finale": [0.789, 0.775, 0.768, None],
    "Modèle": ["Forêt réglée", "Gradient boosting", "Logistique", ""],
}).dropna()

groupe.set_index("Binôme")[["Première", "Finale"]].plot(kind="bar", figsize=(7, 4))
plt.axhline(0.76555, color="gray", linestyle="--", label="Règle bête (0.766)")
plt.ylim(0.7, 0.85)
plt.ylabel("Score Kaggle")
plt.title("Scores du groupe : première vs finale")
plt.xticks(rotation=0)
plt.legend()
plt.show()

## 7. Préparer la présentation de 5 minutes

Chaque binôme présente sa démarche. Pas de slides obligatoires : le notebook et 1 ou 2 graphiques suffisent. Le gabarit :

| Partie | Durée | Contenu |
|---|---|---|
| 1. Nos hypothèses | 1 min | les 2 hypothèses qui vous ont le plus marqués, avec un chiffre chacune |
| 2. Ce qu'on a essayé | 2 min | les variables créées, les modèles comparés, le réglage qui a le plus changé le score, une chose qui n'a **pas** marché |
| 3. Notre score | 1 min | première soumission → finale, où on se situe par rapport à la règle bête et au groupe |
| 4. Ce qu'on referait | 1 min | si on avait 2 heures de plus : quelle variable, quel modèle, quel piège éviter |

Conseils : un seul graphique à l'écran à la fois, chacun parle 2 min 30, et terminez par une phrase que n'importe qui comprendrait (« notre modèle devine 8 fois sur 10 qui survit, surtout grâce au sexe et à la classe »).

In [ ]:
# Remplissez votre plan de présentation, puis exécutez : il s'affiche proprement
presentation = {
    "1. Nos hypothèses": [
        "Les femmes survivent 4 fois plus que les hommes (74 % contre 19 %)",
        "...",
    ],
    "2. Ce qu'on a essayé": [
        "Variables : Famille, Seul, Titre + ...",
        "Modèles : ... le meilleur en validation croisée était ...",
        "Ce qui n'a pas marché : ...",
    ],
    "3. Notre score": [
        f"Première soumission : {SCORE_PREMIERE_SOUMISSION} → finale : {SCORE_SOUMISSION_FINALE}",
    ],
    "4. Ce qu'on referait": [
        "...",
    ],
}

for partie, points in presentation.items():
    print(partie)
    for point in points:
        print("   -", point)
    print()

## À retenir

- Comparer des modèles = **même validation croisée** pour tous, résultats dans un tableau, et regarder l'écart-type autant que la moyenne.
- L'**importance des variables** explique *pourquoi* un modèle décide : c'est l'argument à montrer au client.
- **Sur-apprentissage** : un modèle trop complexe apprend par cœur. Le signe : train qui monte, validation qui descend.
- **Hyperparamètres** : les boutons qu'on règle avant d'entraîner. `GridSearchCV` teste toutes les combinaisons proprement.
- **Fuite de données** : si une colonne contient la réponse, le score est faux. Un score trop beau est un signal d'alarme, pas une victoire.
- Toujours battre la **règle bête** (0.766 sur Kaggle) avant de se réjouir.
- Sur le Titanic, 0.77-0.82 est excellent. Le score n'est pas tout : la **démarche** que vous savez expliquer compte autant.

## Pour montrer aux autres

Les 5 minutes par binôme suivent le gabarit de la section 7. Trois questions guides pour le public :
1. Quelle est la différence entre le score qu'ils annoncent en validation croisée et leur score Kaggle ? Pourquoi ?
2. Quel réglage ou quelle variable a fait la plus grosse différence pour eux ?
3. Qu'est-ce qu'ils feraient différemment avec 2 heures de plus ?

## Liens
- Le classement : https://www.kaggle.com/competitions/titanic/leaderboard
- Les notebooks des autres participants (idées de variables) : https://www.kaggle.com/competitions/titanic/code
- Sur-apprentissage expliqué en images : https://mlu-explain.github.io/bias-variance/
- Pour mettre le notebook sur GitHub : Fichier → Enregistrer une copie sur GitHub (dans Colab)